<a href="https://colab.research.google.com/github/zencolab/colab/blob/main/tts2.5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 验证是否为 NVIDIA L4 (24GB VRAM)
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

# 2. 克隆官方仓库并进入目录
%cd /content
!git clone https://github.com/index-tts/index-tts.git
%cd /content/index-tts

# 3. 安装 uv、高频算子编译工具 ninja 以及 HuggingFace 高速下载器
!pip install -U uv ninja "huggingface_hub[cli,hf_transfer]"

# 4. 同步 WebUI 与 GPT2/FlashAttention 加速引擎依赖 (--extra accel)
!uv sync --extra webui --extra accel

name, memory.total [MiB], compute_cap
NVIDIA L4, 23034 MiB, 8.9
/content
Cloning into 'index-tts'...
remote: Enumerating objects: 1990, done.
remote: Counting objects: 100% (332/332), done.
remote: Compressing objects: 100% (141/141), done.
remote: Total 1990 (delta 258), reused 191 (delta 191), pack-reused 1658 (from 3)
Receiving objects: 100% (1990/1990), 36.72 MiB | 20.66 MiB/s, done.
Resolving deltas: 100% (980/980), done.
/content/index-tts
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 169.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.2/855.2 kB 49.0 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.33.0
    Uninstalling huggingface_hub-1.33.0:
      Successfully uninstalled huggingface_hub-1.33.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [huggingface_hub]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of

In [2]:
# 1. 开启多线程极速下载 IndexTTS-2.5 完整权重
!HF_HUB_ENABLE_HF_TRANSFER=1 hf download IndexTeam/IndexTTS-2.5 --local-dir=checkpoints

# 2. 预下载官方参考音色与情绪音频样本（填充 examples/ 目录）
!uv run python -c "from indextts.utils.examples_downloader import ensure_examples_available; ensure_examples_available()"

/usr/local/lib/python3.13/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 22 files:   0% 0/22 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Reconstructing (incomplete total...):   0% 0.00/2.86k [00:00<?, ?B/s]         

Fetching 22 files:   5% 1/22 [00:00<00:05,  3.72it/s]
Reconstructing (incomplete total...):   0% 2.86k/3.26G [

In [ ]:
%cd /content/index-tts
!uv pip install flash-attn --no-build-isolation

/content/index-tts
Using Python 3.13.16 environment at: /usr
Resolved 31 packages in 593ms
⠋ Preparing packages... (0/1)                                                   

In [ ]:
%cd /content/index-tts

# 1. 还原 webui.py，清除之前 sed 修改可能留下的语法冲突
!git checkout webui.py

# 2. 使用 Python 脚本安全修改 webui.py 并自动匹配支持的加速参数
with open("run_colab_webui.py", "w", encoding="utf-8") as f:
    f.write('''import re
import subprocess
import sys

# 读取原版 webui.py
with open("webui.py", "r", encoding="utf-8") as fp:
    code = fp.read()

# 安全开启 share=True：如果已有 share=... 则替换，否则在 demo.launch( 后注入
if re.search(r"demo\\.launch\\([^)]*\\bshare\\s*=", code):
    code = re.sub(r"\\bshare\\s*=\\s*[^,)]+", "share=True", code)
else:
    code = code.replace("demo.launch(", "demo.launch(share=True, ")

with open("webui.py", "w", encoding="utf-8") as fp:
    fp.write(code)

# 获取当前版本 webui.py 实际支持的命令行参数列表
help_out = subprocess.run(
    [sys.executable, "webui.py", "--help"],
    capture_output=True, text=True
).stdout

cmd = [sys.executable, "webui.py"]

# 按需添加当前 webui.py 支持的参数
if "--model_dir" in help_out:
    cmd.extend(["--model_dir", "./checkpoints"])
if "--bf16" in help_out:
    cmd.append("--bf16")
elif "--fp16" in help_out:
    cmd.append("--fp16")
if "--cuda_kernel" in help_out:
    cmd.append("--cuda_kernel")
if "--qwen_emo" in help_out:
    cmd.append("--qwen_emo")
if "--share" in help_out:
    cmd.append("--share")

# 仅在 flash_attn 可用且 webui.py 支持时开启 --accel
if "--accel" in help_out:
    try:
        import flash_attn
        cmd.append("--accel")
        print("✅ 检测到 flash_attn，已启用 --accel 加速")
    except ImportError:
        print("⚠️ 当前环境未安装 flash_attn，已自动跳过 --accel（不影响正常推理）")

print("🚀 最终执行命令:", " ".join(cmd))
subprocess.run(cmd)
''')

# 3. 启动 WebUI
!GRADIO_SHARE=1 uv run python run_colab_webui.py

/content/index-tts
Updated 0 paths from the index
✅ 检测到 flash_attn，已启用 --accel 加速
🚀 最终执行命令: /content/index-tts/.venv/bin/python webui.py --model_dir ./checkpoints --fp16 --cuda_kernel --qwen_emo --share --accel
>> GPT weights restored from: ./checkpoints/gpt.pth
acceleration engine initialized
GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'

L4 满血版 Python 脚本推理（多语言 + 文本情绪 + 批量缓存）

In [5]:
with open("colab_l4_infer.py", "w", encoding="utf-8") as f:
    f.write('''import time
from indextts.infer_v2_5 import IndexTTS2

print(">> 正在初始化 IndexTTS-2.5 (L4 满血加速模式)...")
tts = IndexTTS2(
    cfg_path="checkpoints/config.yaml",
    model_dir="checkpoints",
    use_bf16=True,           # 启用 L4 原生硬件 BFloat16 Tensor Cores
    use_accel=True,          # 启用 FlashAttention + Paged KV Cache 自回归加速引擎
    use_cuda_kernel=True,    # 启用 BigVGAN 融合 CUDA 激活算子
    use_torch_compile=True,  # 启用 PyTorch 2 Inductor 编译加速 s2mel 流匹配模块
    use_qwen_emo=True        # 24GB 显存充裕，常驻加载 QwenEmotion 文本情绪模型
)

ref_audio = "examples/voice_01.wav"  # 替换为你的 3~10 秒清晰干声路径

# 示例 1：中文零样本克隆 + 多音字精准控制 + 语速控制
t0 = time.time()
tts.infer(
    spk_audio_prompt=ref_audio,
    text="大家好，我现在运行在 Colab Pro 的 L4 显卡上！他在银<行|XING2>里<行|HANG2>走了半天，发现推理速度比之前快太多了。",
    lang="ZH",
    output_path="l4_output_zh.wav",
    duration_factor=1.0,     # 语速因子：0.5~2.0，小于 1.0 加快，大于 1.0 放慢
    verbose=True
)
print(f"✅ 首次合成耗时（含编译预热）: {time.time() - t0:.2f}s")

# 示例 2：利用 QwenEmotion 通过自然语言描述控制情绪（复用参考音频缓存，速度极快）
t1 = time.time()
tts.infer(
    spk_audio_prompt=ref_audio,
    text="终于把所有的加速算子都跑通了，这合成音质和表现力简直太惊艳了！",
    lang="ZH",
    output_path="l4_output_emo.wav",
    use_emo_text=True,
    emo_text="语气充满惊喜、兴奋且语调上扬",
    emo_alpha=0.6,           # 官方建议 0.6 左右兼顾情感表现力与音色自然度
    use_random=False,
    verbose=True
)
print(f"✅ 二次合成耗时（编译后 + 音色缓存命中）: {time.time() - t1:.2f}s")

# 示例 3：跨语言合成（使用同一个中文参考音色说英文/日文）
tts.infer(
    spk_audio_prompt=ref_audio,
    text="With the NVIDIA L4 GPU, real-time factor drops significantly while maintaining high speaker similarity.",
    lang="EN",               # 支持 "ZH", "EN", "JA", "ES", "AR"
    output_path="l4_output_en.wav",
    verbose=True
)
''')

# 运行推理脚本
!PYTHONPATH="$PYTHONPATH:." uv run python colab_l4_infer.py

# 在 Colab 中直接内嵌播放生成的三段音频
from IPython.display import Audio, display
print("🎵 1. 基础中文与多音字控制：")
display(Audio("l4_output_zh.wav"))
print("🎵 2. QwenEmotion 文本情绪控制：")
display(Audio("l4_output_emo.wav"))
print("🎵 3. 跨语言英文合成：")
display(Audio("l4_output_en.wav"))

>> 正在初始化 IndexTTS-2.5 (L4 满血加速模式)...
>> GPT weights restored from: checkpoints/gpt.pth
acceleration engine initialized
GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
W1024 10:23:27.252000 2824 .venv/lib/pytho

🎵 2. QwenEmotion 文本情绪控制：


🎵 3. 跨语言英文合成：
